In [1]:
import pandas as pd 
import random as rand
import numpy as np
import twang as tw

In [2]:
# comando para importar a base de dados 
# converter a coluna de datas em datetime 
# filtrar a faixa temporar de interesse, usada pelos autores: 01/02/2021 - 14/02/2026

df = pd.read_csv("events.csv")
df["event_time"] = df["event_time"].str.removesuffix("UTC")
df["event_time"] = pd.to_datetime(df["event_time"])
df["mont"] = df["event_time"].dt.month
df["day"] = df["event_time"].dt.day
df1 = df[(df["mont"]==2) & (df["day"]>=1) & (df["day"]<=14)]


In [3]:
# Junção dos dados por usuário e por seção 
# Comando pouco utilizado
# criado com a finalidade de verificar se o pré tratamento dos dados está correto 

prep = df1.groupby(by = ["user_id", "user_session"], as_index = False).agg(min_time = ("event_time", "min"), max_time = ("event_time", "max"), events = ("event_type", "count"),
                                                                           products = ("product_id", "unique"), categories = ("category_id", "unique"))
prep["session_time"] = prep["max_time"] - prep["min_time"]
prep["unique_products"] = [len(i) for i in prep["products"]]
prep["unique_categories"] = [len(i) for i in prep["categories"]]

In [4]:
# manipulação da base de dados originais para criar o dataset conforme descrito pelos autores 
# "total_events": quantidade total de eventos por usuário 
# "unique_products": quantidade de produtos com os quais o usuário interagiu, 2 interações com o mesmo produto contam como 1
# "unique_categories": quantidade de categorias de produtos com as quais o usuário interagiu, 2 interações com a mesma categoria contam como 1 
# "active hours": espaço de tempo entre o primeiro e o último evento 
# "view_count": quantidade de views, repetidos contam 
# "cart_count": quantidade de adições ao carrinho 
# "session_count": quantidade de seções por usuário 
# "avg_session_duration": tempo médio de seção por usuário ("active_hours"/"session_count")
# "cart_event_ratio": porcentagem de eventos do tipo "adição ao carrinho" por usuário
# "cart_per_product": proporção de adições ao carrinho por produtos vistos 
# "purchase": classificação final do usuário. Se o usuário realizou uma compra no periodo analizado, ele é classificado como "purchase", do contrário é classificado como "non_purchase"


df2 = pd.crosstab(df1["user_id"], df1["event_type"])
df2["total_events"] = df2["cart"] + df2["purchase"] + df2["view"]

df2["unique_products"] = [len(i) for i in df1.groupby("user_id")["product_id"].unique()]
df2["unique_categories"] = [len(i) for i in df1.groupby("user_id")["category_id"].unique()]

df2["active_hours"] = (prep.groupby("user_id")["max_time"].max() - prep.groupby("user_id")["min_time"].min()).dt.total_seconds()/3600

df2["view_count"] = df2["view"]
df2["cart_count"] = df2["cart"]

df2["session_count"] = [len(i) for i in df1.groupby("user_id")["user_session"].unique()]

df2["avg_session_duration"] = (df2["active_hours"]/df2["session_count"])*60
df2["cart_event_ratio"] = df2["cart_count"]/df2["total_events"]
df2["cart_per_product"] = df2["cart_count"]/df2["unique_products"]

df2["purchase"] = ["purchase" if i >=1 else "non_purchase" for i in df2["purchase"]]
df2.drop(["cart","view"], axis = 1, inplace = True)
df2.reset_index(drop = True, inplace = True)
df2

event_type,purchase,total_events,unique_products,unique_categories,active_hours,view_count,cart_count,session_count,avg_session_duration,cart_event_ratio,cart_per_product
0,non_purchase,7,2,1,111.477778,7,0,1,6688.666667,0.000000,0.0
1,non_purchase,4,4,3,179.837500,4,0,1,10790.250000,0.000000,0.0
2,non_purchase,27,13,4,199.203611,27,0,5,2390.443333,0.000000,0.0
3,non_purchase,2,1,1,168.010000,2,0,1,10080.600000,0.000000,0.0
4,non_purchase,18,1,1,271.352222,18,0,1,16281.133333,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...
39825,non_purchase,1,1,1,0.000000,1,0,1,0.000000,0.000000,0.0
39826,non_purchase,1,1,1,0.000000,1,0,1,0.000000,0.000000,0.0
39827,non_purchase,1,1,1,0.000000,1,0,1,0.000000,0.000000,0.0
39828,non_purchase,2,2,1,0.006667,2,0,1,0.400000,0.000000,0.0


In [5]:
# ajuste da proporção entre usuários "purchase" e "non_purchase" em 4:1 


grupo1 = df2[df2["purchase"] == "purchase"].index.tolist()
grupo2 = rand.sample(df2[df2["purchase"] == "non_purchase"].index.tolist(), len(grupo1)*4)

# divisão do set ajustado em 10 subsets 
set_total = np.hstack([grupo1, grupo2])
dict = {"purchase": grupo1, "non_purchase": grupo2}
lista = list([] for i in range(0,10))
for c in dict:
    np.random.shuffle(dict[c])
    for idx,i in enumerate(dict[c]):
        lista[idx%10].append(i)

# seleção randômica de 3 subsets para teste e 7 para treino 
sort = rand.sample(range(0,10), 7)
lista_treino = []
lista_teste = []

for i in range(0,10):
    if i in sort:
        lista_treino = np.hstack([lista_treino, lista[i]])
    else:
        lista_teste = np.hstack([lista_teste, lista[i]])

In [ ]:
# consolidação dos grupos de teste e de treino e armazenamento dos datasets em .csv
treino = df2.loc[lista_treino]
teste = df2.loc[lista_teste]
treino.to_csv("set_treino.csv")
teste.to_csv("set_teste.csv")

In [81]:
treino = pd.read_csv("set_treino.csv")
treino.drop("Unnamed: 0", axis = 1, inplace = True)